# Zero-shot VQA Inference with vLLM

Batched zero-shot inference (no RAG, no few-shot demos). One image + one
question per prompt, large vLLM batches.

**Model options:**
- A pretrained VLM from Hugging Face (Gemma 3, Qwen-VL) → vLLM loads it directly
- An Unsloth-finetuned LoRA → first export with `save_pretrained_merged`
  (Section 0), then pass that folder to vLLM. vLLM **cannot** load an Unsloth
  4-bit checkpoint directly.

## 0. (Optional) Export an Unsloth fine-tuned model for vLLM

Run this section **only** if `MODEL_PATH` below points at an Unsloth LoRA.
Restart the runtime after exporting, then continue from Section 1.

In [ ]:
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted.')

In [ ]:
%%capture
!pip install unsloth

In [ ]:
from unsloth import FastVisionModel

UNSLOTH_LORA = "/content/drive/My Drive/licenta/new_llm/qwen-big-retranlsate-100k"
EXPORT_DIR   = "/content/model_vllm_merged"   # local disk = fast

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = UNSLOTH_LORA,
    load_in_4bit = True,
)
model.save_pretrained_merged(EXPORT_DIR, tokenizer)
print(f"Exported locally to {EXPORT_DIR}. Now restart runtime and continue from Section 1.")

## 1. Install vLLM

In [ ]:
%%capture
!pip install -q vllm pillow scikit-learn
!pip uninstall -y torchcodec 2>/dev/null

In [ ]:
import os
os.environ["VLLM_USE_FLASHINFER_SAMPLER"]    = "0"
os.environ["TRANSFORMERS_NO_TORCHCODEC"]     = "1"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

## 2. Mount Drive + config

In [ ]:
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted.')

In [ ]:
# Pass either the exported merged folder (from Section 0) or a pretrained HF id, e.g.
#   "Qwen/Qwen2.5-VL-7B-Instruct"
#   "google/gemma-3-27b-it"
MODEL_PATH = "/content/model_vllm_merged"

# VQAv2 Romanian test set
TEST_JSON       = "/content/drive/MyDrive/VQA_Val_Subset_15k/val_subset_15k_retranslated_gemma4_bigmodel.json"
COCO_IMAGES_ZIP = "/content/drive/MyDrive/VQA_Val_Subset_15k/val_subset_15k_archive.zip"

SAVE_PATH  = "/content/drive/My Drive/licenta/predictions/vllm_zeroshot_vqav2_qwen.jsonl"
VLLM_BATCH = 1000

os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

## 3. Unzip COCO images

In [ ]:
import glob
os.makedirs("/content/data/coco", exist_ok=True)
!unzip -q -o "$COCO_IMAGES_ZIP" -d /content/data/coco

COCO_IMAGE_DIR = None
for c in ["/content/data/coco/images", "/content/data/coco/val2014", "/content/data/coco"]:
    if os.path.isdir(c) and glob.glob(os.path.join(c, "*.jpg")):
        COCO_IMAGE_DIR = c
        break
if COCO_IMAGE_DIR is None:
    COCO_IMAGE_DIR = os.path.dirname(
        glob.glob("/content/data/coco/**/*.jpg", recursive=True)[0]
    )
print(f"COCO images: {len(glob.glob(os.path.join(COCO_IMAGE_DIR, '*.jpg')))} in {COCO_IMAGE_DIR}")

## 4. Image loader + answer cleaning

In [ ]:
from PIL import Image
import re

def load_coco_img(image_id, image_filename=None):
    if image_filename:
        p = os.path.join(COCO_IMAGE_DIR, image_filename)
        if os.path.exists(p):
            try: return Image.open(p).convert("RGB")
            except Exception: pass
    p = os.path.join(COCO_IMAGE_DIR, f"COCO_val2014_{int(image_id):012d}.jpg")
    if os.path.exists(p):
        try: return Image.open(p).convert("RGB")
        except Exception: return None
    return None

def clean_answer(text):
    text = text.strip().split("\n")[0].strip()
    text = re.sub(r"(?i)^.*?r[ăa]spuns\s*:?\s*", "", text).strip()
    text = text.split(" ")[0].strip()
    text = re.sub(r"[^\wăâîșțĂÂÎȘȚ]", "", text)
    return text.lower()

## 5. Start vLLM

In [ ]:
# Make stdout/stderr have a real fileno() — vLLM needs it.
import sys
class _StdoutFileno:
    def __init__(self, stream, fd):
        self._stream = stream
        self._fd     = fd
    def __getattr__(self, n):
        return getattr(self._stream, n)
    def fileno(self):
        return self._fd

sys.stdout = _StdoutFileno(sys.stdout, 1)
sys.stderr = _StdoutFileno(sys.stderr, 2)

from vllm import LLM, SamplingParams
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_PATH, fix_mistral_regex=True)
llm = LLM(
    model = MODEL_PATH,
    dtype = "bfloat16",
    max_model_len = 2048,
    gpu_memory_utilization = 0.80,
    limit_mm_per_prompt = {"image": 1},
    trust_remote_code = True,
)

## 6. Load test set

In [ ]:
import json
with open(TEST_JSON, encoding="utf-8") as f:
    test_raw = json.load(f)

test_items = []
for v in test_raw:
    test_items.append({
        "questionId":     str(v["question_id"]),
        "imageId":        str(v["imageId"]),
        "image_filename": v.get("image_filename", ""),
        "question":       v.get("question_retranslated") or v.get("question", ""),
        "gt_answer":      v.get("answer_retranslated")   or v.get("answer", ""),
    })
print(f"Test examples: {len(test_items)}")

## 7. Zero-shot prompt (image + question)

In [ ]:
INSTRUCTION = "Răspunde la întrebare cu un singur cuvânt în română."

def build_request(question, image):
    messages = [{"role": "user", "content": [
        {"type": "text",  "text": INSTRUCTION},
        {"type": "image"},
        {"type": "text",  "text": f"Întrebare: {question} Răspuns:"},
    ]}]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True,
                                           tokenize=False)
    return {"prompt": prompt, "multi_modal_data": {"image": [image]}}

sampling_params = SamplingParams(temperature=0.0, max_tokens=30, stop=["\n"])

## 8. Batched inference (resume-safe)

In [ ]:
from tqdm.auto import tqdm

processed = set()
if os.path.exists(SAVE_PATH):
    with open(SAVE_PATH, encoding="utf-8") as f:
        for line in f:
            try: processed.add(json.loads(line)["questionId"])
            except Exception: pass
print(f"Already processed: {len(processed)}")

todo = [ex for ex in test_items if ex["questionId"] not in processed]
print(f"To process: {len(todo)}")

with open(SAVE_PATH, "a", encoding="utf-8") as out_f:
    for bs in tqdm(range(0, len(todo), VLLM_BATCH), desc="vLLM batches"):
        batch = todo[bs:bs + VLLM_BATCH]
        vllm_inputs, meta = [], []
        for ex in batch:
            img = load_coco_img(ex["imageId"], ex.get("image_filename"))
            if img is None:
                continue
            vllm_inputs.append(build_request(ex["question"], img))
            meta.append(ex)

        if not vllm_inputs:
            continue

        outputs = llm.generate(vllm_inputs, sampling_params)
        for ex, out in zip(meta, outputs):
            pred = out.outputs[0].text.strip()
            out_f.write(json.dumps({
                "questionId": ex["questionId"],
                "imageId":    ex["imageId"],
                "question":   ex["question"],
                "gt_answer":  ex["gt_answer"],
                "prediction": pred,
            }, ensure_ascii=False) + "\n")
        out_f.flush()

print("Done.")

## 9. Evaluate

In [ ]:
import unicodedata
from sklearn.metrics import accuracy_score, f1_score

def clean_text(t):
    if not t:
        return ""
    t = unicodedata.normalize("NFC", str(t).lower().strip())
    return re.sub(r"\s+", " ", t).strip().strip(".")

# Romanian gender/number canonicalization for colors and adjectives
CANON = {
    "albă":"alb","albe":"alb","albi":"alb","albul":"alb",
    "neagră":"negru","negre":"negru","negri":"negru","negrul":"negru",
    "roșie":"roșu","roșii":"roșu","rosu":"roșu","rosie":"roșu","roșu":"roșu",
    "albastră":"albastru","albastre":"albastru","albaștri":"albastru",
    "verzi":"verde",
    "galbenă":"galben","galbene":"galben","galbeni":"galben",
    "portocalie":"portocaliu","portocalii":"portocaliu",
}
def normalize(t):
    t = clean_text(t)
    return " ".join(CANON.get(w, w) for w in t.split())

preds = []
with open(SAVE_PATH, encoding="utf-8") as f:
    for line in f:
        try: preds.append(json.loads(line))
        except Exception: continue

gt   = [normalize(p["gt_answer"])  for p in preds]
pred = [normalize(p["prediction"]) for p in preds]

print(f"Examples: {len(preds)}")
print(f"Accuracy (normalized): {accuracy_score(gt, pred):.4f}")
print(f"F1 weighted:           {f1_score(gt, pred, average='weighted'):.4f}")

## Notes

- **Unsloth fine-tuned model:** export first (Section 0) with `save_pretrained_merged`,
  restart the runtime, then point `MODEL_PATH` at the exported folder
- **Pretrained model:** use the full-precision repo (e.g. `google/gemma-3-27b-it`,
  `Qwen/Qwen2.5-VL-7B-Instruct`) — **not** the `-bnb-4bit` variant
- **Speed:** without RAG and with one image per prompt, batches of 1000+ are fine.
  On A100 the 15k VQAv2 subset finishes in a few minutes
- **Comparison:** run this notebook (zero-shot) alongside `04` (RAG few-shot) for
  an ablation